# 11 · Self-Correction, Auto-Retry & JSON Repair

Agents fail in small, recoverable ways: a flaky tool, malformed JSON, a missing field. This notebook handles them without giving up — `with_retry` re-runs flaky calls, `repair_json` fixes broken model output against a schema, and `self_correct` feeds a validation error back so the model tries again.

> Runs on Ollama.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'agents_lab').exists())
sys.path.insert(0, str(root))

**Auto-retry** a callable that fails the first couple of times (e.g. a transient network blip).

In [ ]:
from agents_lab import with_retry

attempts = {'n': 0}
def flaky():
    attempts['n'] += 1
    if attempts['n'] < 3:
        raise ConnectionError('transient')
    return 'success'

print(with_retry(flaky, max_attempts=5), 'after', attempts['n'], 'tries')

**JSON repair:** valid JSON parses instantly; broken JSON is sent back to the model to fix against a schema.

In [ ]:
from agents_lab import repair_json

print(repair_json('{"a": 1, "b": 2}'))                       # parses directly
schema = {'type': 'object', 'properties': {'name': {'type': 'string'}, 'age': {'type': 'integer'}}}
print(repair_json("name = Bob, age = 30", schema=schema, provider='ollama'))  # model repairs

**Self-correction:** validate an answer; on failure, feed the error back and retry until it passes.

In [ ]:
from agents_lab import self_correct

def validator(ans):
    ok = ans.strip().isdigit() and int(ans.strip()) % 2 == 0
    return ok, 'answer must be a single even integer with no other text'

print(self_correct('Give me an even number between 10 and 20.', validator, provider='ollama'))

**Takeaway:** robust agents expect failure and recover — retry transient errors, repair malformed output, and self-correct against validators.

Next → `12_loop_control.ipynb`